# Project Setup - Mount Google Drive

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q langchain langchain-groq langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


# Install LangChain and Groq Dependencies

In [ ]:
!pip install -q faiss-cpu langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.0 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq

print("Groq working!")

Groq working!


# Agent 1: Symptom Extraction Agent
### Purpose:
- Initialize LLM (Groq)
- Process raw patient text
- Extract structured symptoms in JSON format

In [ ]:
import os
import json

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate  # Fixed import path

# API Key (Make sure to update this if you rotate your key!)
os.environ["GROQ_API_KEY"] = "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"

# LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1
)

# Function
def extract_symptoms(user_input):
    prompt = PromptTemplate.from_template("""
You are an expert medical NLP assistant.

Extract symptoms from the user message.

Return ONLY valid JSON in this exact format. Do not add any extra text, explanations, or markdown.

{{
  "symptoms": [
    {{
      "symptom": "headache",
      "severity": "severe",
      "duration": "3 days",
      "body_part": "head"
    }}
  ]
}}

User Input: {user_input}
""")

    # Modern LCEL syntax (Fixes the LLMChain deprecation error)
    chain = prompt | llm

    # Run the chain
    response = chain.invoke({"user_input": user_input})
    result = response.content # Extract the text content from the message object

    # Clean the output
    cleaned = result.strip().replace("```json", "").replace("```", "").strip()

    try:
        parsed = json.loads(cleaned)
        return parsed
    except Exception as e:
        return {
            "error": "Failed to parse JSON",
            "raw_output": result,
            "cleaned": cleaned,
            "details": str(e)
        }

# Test again
output = extract_symptoms("I have severe headache and high fever for 3 days with body pain.")
print(json.dumps(output, indent=2))

{
  "symptoms": [
    {
      "symptom": "headache",
      "severity": "severe",
      "duration": "3 days",
      "body_part": "head"
    },
    {
      "symptom": "fever",
      "severity": "high",
      "duration": "3 days",
      "body_part": "whole body"
    },
    {
      "symptom": "body pain",
      "severity": null,
      "duration": "3 days",
      "body_part": "whole body"
    }
  ]
}


# Medical Knowledge Base Creation (RAG Setup)
### Purpose:
- Load medical dataset
- Split documents into chunks
- Generate embeddings
- Create FAISS Vector Database

In [ ]:
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import os

def load_medical_vector_db(data_path,index_path):

    # Load CSV documents
    loader = CSVLoader(file_path=data_path)
    documents = loader.load()

    print(f"Loaded {len(documents)} documents")

    # Split into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=50)

    texts = text_splitter.split_documents(documents)

    print(f"Created {len(texts)} text chunks")

    # Embedding model
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    # Create FAISS vector DB
    vectorstore = FAISS.from_documents(
        texts,
        embeddings
    )

    # Save index
    vectorstore.save_local(index_path)

    print("✅ Vector Database Ready!")

    return vectorstore


# Run
vector_db = load_medical_vector_db(
    data_path="/content/drive/MyDrive/symptom-checker/Symptor_checker_dnlp_project/data/dataset.csv",

    index_path="/content/drive/MyDrive/symptom-checker/Symptor_checker_dnlp_project/medical_faiss_index"
)

/tmp/ipykernel_2463/3561579466.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


Loaded 4920 documents
Created 5268 text chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector Database Ready!


# Week 1 Testing Cell
### Purpose:
- Test symptom extraction
- Verify vector database creation

In [ ]:
# Test cell
# Final Week 1 Test
print("=== Week 1 Test ===")
symptoms = extract_symptoms("I have been dealing with a sore throat and body aches for 4 days.")
print("Symptoms:", json.dumps(symptoms, indent=2))

print("\nVector DB is ready:", vector_db is not None)

=== Week 1 Test ===
Symptoms: {
  "symptoms": [
    {
      "symptom": "sore throat",
      "severity": "moderate",
      "duration": "4 days",
      "body_part": "throat"
    },
    {
      "symptom": "body aches",
      "severity": "moderate",
      "duration": "4 days",
      "body_part": "whole body"
    }
  ]
}

Vector DB is ready: True


# Agent 2: Medical Context Retrieval Agent (RAG)
### Purpose:
- Search FAISS vector database
- Retrieve relevant medical documents
- Provide context for disease prediction

In [ ]:
from langchain_community.vectorstores import FAISS # this import faiss.
from langchain_huggingface import HuggingFaceEmbeddings # import embedding models( convert text ->numbers/vectors)
def retrive_medical_context(symptoms_dict,k=5):
  # load embedding model
  embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") # main job : Convert sentences into vector.
  # load the vector database
  vectorstore=FAISS.load_local("/content/drive/MyDrive/symptom-checker/Symptor_checker_dnlp_project/medical_faiss_index",embeddings,allow_dangerous_deserialization=True)
  # convert symptoms into query
  query=json.dumps(symptoms_dict) # this convert dict into text
  # Retrice Top -k relevent documents
  docs=vectorstore.similarity_search(query,k=k) # meaning find top 5 relevent medical document relevent to ( lets say headace + fever)
  # Format documents
  context=""
  sources=[]
  for i,doc in enumerate(docs):
    context+=f"\n--- Document {i+1} ---\n{doc.page_content}\n"
    sources.append(doc.metadata.get('source','Unknown'))
  return {
      "context":context.strip(),
      "sources":sources,
      "num_docs":len(docs)
  }
# Testing Agent-2
test_symptoms={
    "symptoms":[
        {"symptom":"headace","severity":"severe"},
        {"symptom":"fever","severity":"high"}
    ]
}
retrived=retrive_medical_context(test_symptoms)
print(f"Retrived {retrived['num_docs']} documsnts")
print("Sources:", retrived['sources'])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrived 5 documsnts
Sources: ['/content/drive/MyDrive/symptom-checker/Symptor_checker_dnlp_project/data/dataset.csv', '/content/drive/MyDrive/symptom-checker/Symptor_checker_dnlp_project/data/dataset.csv', '/content/drive/MyDrive/symptom-checker/Symptor_checker_dnlp_project/data/dataset.csv', '/content/drive/MyDrive/symptom-checker/Symptor_checker_dnlp_project/data/dataset.csv', '/content/drive/MyDrive/symptom-checker/Symptor_checker_dnlp_project/data/dataset.csv']


# Agent 3: Disease Prediction Agent
### Purpose:
- Analyze symptoms
- Use retrieved medical context
- Predict possible diseases


In [ ]:
import json
from langchain_core.prompts import PromptTemplate
def predict_disease(symptoms_dict,retrieved_context):
  # symptoms entered by user+ medical info pulled from database (RAG output)
  # step-1 creating prompt template
  prompt=PromptTemplate.from_template("""
  This is building AI instruction (prompt)
  we tell LLM model how you should behave
  We tell Ai act like a medical assistant (not a doctor)
  So we make rules based on symptoms and the retrived medical context below.
  Rule1: Only use information from Retrived context ( Only use FAISS/RAG results)
  Rule2: Say 'possible' or 'may suggest'
  Rule3: Give 2-4 most likely conditions with brief reason.
  Rule4: If emergency symptoms,mention clearly.
  Extracted Symptoms: {symptoms}
  Retrieved Medical Context: {context}
  Output Fomat Instruction :
  Return in simple bullet points.
  """)
  # Creating a LLM chain
  chain = prompt | llm # llm= language model( GPT/Huggingface)
                                        # Prompt= Instruction we just created
  response = chain.invoke({
        "symptoms": json.dumps(symptoms_dict, indent=2),
        "context": retrieved_context["context"]
    })
  return response.content
# Test Agent 3
prediction = predict_disease(test_symptoms, retrived)
print(prediction)









Based on the extracted symptoms and retrieved medical context, here are the possible conditions:

* Malaria may suggest, as the symptoms of severe headache and high fever are consistent with the disease, which is characterized by high fever and headache in the retrieved medical context.
* Viral infection may suggest, as the symptoms of headache and fever can also be associated with viral infections, although the retrieved context primarily points towards malaria.
* Bacterial infection may suggest, as severe headache and high fever can also be symptoms of bacterial infections, but the retrieved context does not provide strong evidence for this condition.
* It is essential to seek medical attention immediately, as high fever and severe headache can be emergency symptoms if left untreated or if they are symptoms of a severe underlying condition.


# Severity Assessment Helper Function
### Purpose:
- Analyze symptom seriousness
- Classify risk level
- Detect emergency conditions

In [ ]:
def classify_severity(symptoms_dict):
  # it looks at severity words and symptoms to decide level.
  severity_score=0
  emergency_keywords=["chest pain", "difficulty breathing", "unconscious", "severe bleeding", "stroke", "high fever"]
  high_keywords=["severe", "intense", "worst", "vomiting blood"]
  text=json.dumps(symptoms_dict).lower()
  # check for emergencey keywords
  for kw in emergency_keywords:
    if kw in text:
      return "EMERGENCY"
  # check for high pirority keywords
  for kw in high_keywords:
    if kw in text:
      return "HIGH"
  # check for moderate conditions
  if "moderate" in text:
    return "MODERATE"
  else:
    if "fever" in text:
      return "MODERATE"
    else:
      return "MILD"
# Testing
print("Severity Test:", classify_severity({
    "symptoms": [{"symptom": "severe headache", "severity": "severe"}]
}))



Severity Test: HIGH


# Agent 4: Medication Guidance Agent
### Purpose:
- Provide OTC medicine suggestions
- Recommend home remedies
- Apply medication safety rules

In [ ]:
def get_medication_guidance(symptoms_dict,prediction,severity_level):
  # Suggest ONLY safe OTC medicine and home remedies.
  # it uses safety prompts + disclaimer.
 prompt = PromptTemplate.from_template("""
You are a safe medication information assistant.

STRICT RULES (never break them):
- ONLY suggest common over-the-counter (OTC) medicines like Paracetamol (Tylenol), Ibuprofen (Advil), etc.
- NEVER suggest antibiotics, prescription drugs, opioids, or controlled substances.
- Always include a strong disclaimer.
- For mild symptoms: prefer home remedies.
- For moderate: suggest OTC with dosage.
- For high/emergency: say "See a doctor immediately" and skip medicine suggestions.

Extracted Symptoms: {symptoms}
Predicted Conditions: {prediction}
Severity Level: {severity}

Provide output in this format:
- Home Remedies: ...
- Suggested OTC (if appropriate): Name, simple dosage, note
- Recommendation: Home / OTC / Doctor
- Disclaimer: ...

Return only the structured text.
""")
 chain=prompt | llm
 response = chain.invoke({
     "symptoms":json.dumps(symptoms_dict,indent=2),
     "prediction": prediction,
     "severity":severity_level
 })
 return response.content
# testing..
test_symptoms = {"symptoms": [{"symptom": "headache", "severity": "severe"}]}
test_prediction = "Malaria or Viral infection may be possible."
print(get_medication_guidance(test_symptoms, test_prediction, "MODERATE"))


- Home Remedies: Rest, stay hydrated by drinking plenty of water, and apply a cool compress to the forehead.
- Suggested OTC (if appropriate): Paracetamol (Tylenol), 500mg every 4-6 hours, to help alleviate headache symptoms.
- Recommendation: OTC
- Disclaimer: The information provided is for general purposes only and is not a substitute for professional medical advice. If symptoms worsen or persist, see a doctor immediately. Always consult a healthcare professional before taking any medication, especially if you have any pre-existing medical conditions or are taking other medications.


# Agent 5: Final Recommendation Agent
### Purpose:
- Combine outputs from all agents
- Generate final patient guidance
- Recommend next actions
- Produce safety warnings

In [ ]:
def generate_recommendations(symptoms_dict,predictions,medication_guidance,severity_level):
  """ Agent 5: Combines everything and gives final safe advice."""
  prompt = PromptTemplate.from_template("""
You are a responsible medical recommendation assistant.

Summarize the following into clear, safe advice:

Symptoms: {symptoms}
Possible Conditions: {prediction}
Medication Guidance: {medication}
Severity: {severity}

Output in bullet points:
- Overall Assessment
- Action Plan (Home care / OTC / See Doctor)
- When to Seek Emergency Help
- Lifestyle Tips
- Final Disclaimer (must include)
""")
  chain=prompt |llm
  response =chain.invoke({
      "symptoms":json.dumps(symptoms_dict),
      "prediction":prediction,
      "medication":medication_guidance,
      "severity":severity_level
  })
  return response.content

# Final Testing and Evaluation

In [ ]:
# Test Agent 5 separately
test_symptoms = {
    "symptoms": [
        {"symptom": "headache", "severity": "severe", "duration": "3 days"}
    ]
}

test_prediction = "Malaria or Viral infection may suggest..."
test_medication = "- Home Remedies: Rest and hydration\n- Suggested OTC: Paracetamol"
test_severity = "HIGH"

result = generate_recommendations(test_symptoms, test_prediction, test_medication, test_severity)
print(result)

* **Overall Assessment**: You are experiencing a severe headache that has lasted for 3 days, which could be a symptom of a serious underlying condition such as malaria, viral infection, or bacterial infection. The severity of your symptoms is considered HIGH.
* **Action Plan**:
  - Home care: Rest and stay hydrated to help manage your symptoms.
  - Over-the-counter (OTC) medication: You can consider taking Paracetamol to help alleviate your headache and fever. However, always follow the recommended dosage and consult with a doctor before taking any medication.
  - See a doctor: It is essential to seek medical attention as soon as possible to determine the underlying cause of your symptoms and receive proper treatment.
* **When to Seek Emergency Help**: If you experience any of the following, seek emergency medical attention immediately:
  - Severe headache that worsens over time
  - High fever that does not respond to medication
  - Confusion, disorientation, or difficulty speaking
  -

# Main Pipeline Controller
### Purpose:
- Execute Agent 1 → Agent 2 → Agent 3 → Agent 4 → Agent 5
- Orchestrate complete workflow

In [ ]:
def run_symptom_pipeline(user_input):
  """ Complete  Pipeline Uptill now : Agents 1 -> 5 """
  print("\n Running Full Multi-Agent PipeLine\n")
  # Agent-1 ( Takes raw input and extract structured symptoms)
  symptoms=extract_symptoms(user_input)
  print("Agent-1 : Symptoms Extracted")

  # Agent-2 ( Taked structured symptoms , searches FAISS vector database and find relevent medical documents)
  retrived=retrive_medical_context(symptoms)
  print(f" Agent-2 Retrived {retrived['num_docs']} medical documents")

  # Agent-3 (Takes Symptoms agent-1 output , taked medical context ( agant-2 output ), sends both to LLM , gets possible disease)
  prediction=predict_disease(symptoms,retrived)
  print("Agent-3 : Disease Prediction Complete")

  # Week-3 additions
  severity=classify_severity(symptoms)
  print(f"Severity Classified : {severity} ")

  medication=get_medication_guidance(symptoms,prediction,severity)
  print(f" Agent-4 : Medication Guidance Ready ")

  final_recommendation=generate_recommendations(symptoms,prediction,medication,severity)
  print(f" Agent-5 : Final Recommendation Ready ")
  # Final Output
  print("="*70)
  print("FINAL Patient Report ")
  print("="*70)
  print("Symptoms:", json.dumps(symptoms, indent=2))
  print("\nPossible Conditions:\n", prediction)
  print("\nMedication Guidance:\n", medication)
  print("\nFinal Recommendations:\n", final_recommendation)
  print("\nSources:", retrived['sources'])
  return {
        "symptoms": symptoms,
        "prediction": prediction,
        "medication": medication,
        "recommendations": final_recommendation,
        "severity":severity
    }
# Final Test
result = run_symptom_pipeline("I have been having severe headache and fever for 3 days with body pain.")




 Running Full Multi-Agent PipeLine

Agent-1 : Symptoms Extracted


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Agent-2 Retrived 5 medical documents
Agent-3 : Disease Prediction Complete
Severity Classified : HIGH 
 Agent-4 : Medication Guidance Ready 
 Agent-5 : Final Recommendation Ready 
FINAL Patient Report 
Symptoms: {
  "symptoms": [
    {
      "symptom": "headache",
      "severity": "severe",
      "duration": "3 days",
      "body_part": "head"
    },
    {
      "symptom": "fever",
      "severity": null,
      "duration": "3 days",
      "body_part": "whole body"
    },
    {
      "symptom": "body pain",
      "severity": null,
      "duration": "3 days",
      "body_part": "whole body"
    }
  ]
}

Possible Conditions:
 Based on the extracted symptoms and retrieved medical context, here are the possible conditions:

* Malaria may suggest, as the symptoms of headache, fever, and body pain (which can be related to muscle pain) are mentioned in the retrieved context.
* Viral infection may suggest, as the symptoms of headache, fever, and body pain are common in various viral infection